In [11]:
from google.colab import drive
drive.mount('/content/drive')

# === Install Google Chrome ===
!apt-get update -qq
!apt-get install -y wget unzip
!wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!apt-get install -y ./google-chrome-stable_current_amd64.deb

# === Get the matching ChromeDriver ===
!CHROME_VERSION=$(google-chrome --version | awk '{print $3}' | cut -d'.' -f1)
!DRIVER_URL=$(wget -qO- https://googlechromelabs.github.io/chrome-for-testing/latest-patch-versions-per-build-with-downloads.json \
  | grep -A20 "\"$CHROME_VERSION\"" \
  | grep "chromedriver-linux64.zip" \
  | head -1 \
  | cut -d'"' -f4)

!echo "✅ Chrome major version: $CHROME_VERSION"
!echo "⬇️ Downloading driver from: $DRIVER_URL"

!wget -q "$DRIVER_URL" -O /tmp/chromedriver.zip
!unzip -o /tmp/chromedriver.zip -d /usr/local/bin/
!chmod +x /usr/local/bin/chromedriver-linux64/chromedriver


!pip install selenium webdriver-manager pillow

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
unzip is already the newest version (6.0-26ubuntu3.2).
wget is already the newest version (1.21.2-2ubuntu1.1).
0 upgraded, 0 newly installed, 0 to remove and 132 not upgraded.
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
Note, selecting 'google-chrome-stable' instead of './google-chrome-stable_current_amd64.deb'
The following additional packages will be installed:
  at-spi2-core gsettings-desktop-schemas libatk-bridge2.0-0 libatk1.0-0
  libatk1.0-data libatspi2.0-0 libvulkan1 libxcomposite1 libxtst6
  mesa-vulkan-drivers session-migration
The following NEW packages will be installed:
  at-spi2-core google-chrome-stable gsettings-desktop-schemas
  libatk-

In [12]:
from PIL import Image
from collections import deque

def remove_background(img: Image.Image):
    img = img.convert("RGBA")
    pixels = img.load()
    width, height = img.size

    BORDER_COLOR = (85, 85, 85)  # #555555

    # Step 1 — make region transparent
    for x in range(width):
        for y in range(height):
            if x <= 246 and y <= 265:
                r, g, b, a = pixels[x, y]
                pixels[x, y] = (r, g, b, 0)

    # Step 2 — flood fill background from edges
    visited = set()
    q = deque()

    # start flood fill from all edges
    for x in range(width):
        q.append((x, 0))
        q.append((x, height - 1))

    for y in range(height):
        q.append((0, y))
        q.append((width - 1, y))

    while q:
        x, y = q.popleft()

        if (x, y) in visited:
            continue
        if not (0 <= x < width and 0 <= y < height):
            continue

        visited.add((x, y))

        r, g, b, a = pixels[x, y]

        # stop when reaching the border color
        if (r, g, b) == BORDER_COLOR:
            continue

        # make background transparent
        pixels[x, y] = (r, g, b, 0)

        # explore neighbors
        q.extend([
            (x+1, y),
            (x-1, y),
            (x, y+1),
            (x, y-1)
        ])

    return img

In [40]:
from PIL import Image
from collections import deque
import math

def remove_background_and_soften_edges(img: Image.Image):
    img = img.convert("RGBA")
    pixels = img.load()
    width, height = img.size

    BORDER = (85, 85, 85)  # #555555
    BORDER_THRESHOLD = 259   # how close a color must be to be considered edge

    # ---- Step 1: remove fixed rectangle ----
    for x in range(width):
        for y in range(height):
            if x <= 246 or y >= 265:
                r,g,b,a = pixels[x,y]
                pixels[x,y] = (r,g,b,0)

    # ---- Step 2: floodfill background ----
    visited = set()
    q = deque()

    for x in range(width):
        q.append((x,0))
        q.append((x,height-1))
    for y in range(height):
        q.append((0,y))
        q.append((width-1,y))

    while q:
        x,y = q.popleft()
        if (x,y) in visited:
            continue
        if not (0 <= x < width and 0 <= y < height):
            continue

        visited.add((x,y))
        r,g,b,a = pixels[x,y]

        if (r,g,b) == BORDER:
            continue

        r,g,b,a = pixels[x,y]

        if a == 0:
            continue

        # color distance from #555555
        dist = math.sqrt(
            (r-BORDER[0])**2 +
            (g-BORDER[1])**2 +
            (b-BORDER[2])**2
          )

        if dist < BORDER_THRESHOLD:
            # snap color to exact border
            alpha = int(255 * (1 - dist / BORDER_THRESHOLD))
            alpha = max(0, alpha)
            pixels[x,y] = (85,85,85,alpha)
        else:
            pixels[x,y] = (r,g,b,0)

        q.extend([(x+1,y),(x-1,y),(x,y+1),(x,y-1)])

    return img

In [41]:
# Step 3: Import libraries
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from PIL import Image
import io
import base64

# Step 4: Define function
from PIL import Image

def save_shape_to_drive(shape_code, shape_name, save_path="/content/drive/MyDrive/"):
    # Setup Chrome options for Colab
    options = webdriver.ChromeOptions()
    options.add_argument("--headless")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")

    # Launch browser
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
    driver.get("https://viewer.shapez.io/")

    # Find input field and set value
    input_box = driver.find_element(By.ID, "code")
    input_box.clear()
    input_box.send_keys(shape_code)

    # Wait for canvas to update
    time.sleep(2)

    # Get canvas element
    canvas = driver.find_element(By.ID, "result")

    # Extract image from canvas using JS
    canvas_png = driver.execute_script(
        "return arguments[0].toDataURL('image/png').substring(22);", canvas
    )

    # Decode and open image
    image_data = base64.b64decode(canvas_png)
    image = Image.open(io.BytesIO(image_data)).convert("RGBA")

    # Make #e9eaec and #ffffff transparent
    image = remove_background_and_soften_edges(image)

    # Save to Drive
    filename = f"{save_path}{shape_name}.png"
    image.save(filename, "PNG")

    print(f"Saved: {filename}")
    driver.quit()


In [42]:
# test
save_shape_to_drive("----Rr--:Cu------", "Cu5", "/content/drive/MyDrive/shapes/")

Saved: /content/drive/MyDrive/shapes/Cu5.png


In [43]:
all_shapes = []
for levels in range(4):
  for i in "CRWS":
    for j in "rgbypcuw":
      code = ("----Rr--:"*levels) + f"{i}{j}------"
      name = i+j+str(levels)
      save_shape_to_drive(code, name, "/content/drive/MyDrive/shapes/")


Saved: /content/drive/MyDrive/shapes/Cr0.png
Saved: /content/drive/MyDrive/shapes/Cg0.png
Saved: /content/drive/MyDrive/shapes/Cb0.png
Saved: /content/drive/MyDrive/shapes/Cy0.png
Saved: /content/drive/MyDrive/shapes/Cp0.png
Saved: /content/drive/MyDrive/shapes/Cc0.png
Saved: /content/drive/MyDrive/shapes/Cu0.png
Saved: /content/drive/MyDrive/shapes/Cw0.png
Saved: /content/drive/MyDrive/shapes/Rr0.png
Saved: /content/drive/MyDrive/shapes/Rg0.png
Saved: /content/drive/MyDrive/shapes/Rb0.png
Saved: /content/drive/MyDrive/shapes/Ry0.png
Saved: /content/drive/MyDrive/shapes/Rp0.png
Saved: /content/drive/MyDrive/shapes/Rc0.png
Saved: /content/drive/MyDrive/shapes/Ru0.png
Saved: /content/drive/MyDrive/shapes/Rw0.png
Saved: /content/drive/MyDrive/shapes/Wr0.png
Saved: /content/drive/MyDrive/shapes/Wg0.png
Saved: /content/drive/MyDrive/shapes/Wb0.png
Saved: /content/drive/MyDrive/shapes/Wy0.png
Saved: /content/drive/MyDrive/shapes/Wp0.png
Saved: /content/drive/MyDrive/shapes/Wc0.png
Saved: /co